# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [57]:
ПРІЗВИЩЕ_ІМЯ = 'Кшик Олена'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-34'
НОМЕР        = 9       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [58]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 9   ·   Кшик Олена   ·   ШІ-34

Ваші вісім стовпців:
   1. Телефон
   2. Стать
   3. ЛН
   4. Поступлення
   5. spO2 з дот
   6. Т вище38
   7. БРпр2
   8. С/Я2

Ваша пара для аналізу зв’язку: spO2 без дот та помер/жив

Ваше додаткове питання:
   Знайдіть у файлі стовпець, у якому порожня клітинка означає не «невідомо», а «норма». Доведіть це числами й поясніть, чому dropna() тут небезпечний.


In [59]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Стать,ЛН,Поступлення,spO2 з дот,Т вище38,БРпр2,С/Я2
0,0958469788 дружина Ганна,ч,ІІІ-IV,2022-02-24,98.0,4 дні до госп,5.1,74.0
1,"0671778486, 0994887663 дружина Оксана",ч,ІІ-ІІІ,2022-02-22,95.0,5 до гос,3.4,79.0
2,"0958768045, чоловік Степан 0951784726",ж,ІІ,2021-12-30,90.0,10 до гос,6.8,89.0
3,0959819331. дочка Валентина 0960845205,ж,ІІІ-IV,2021-12-24,96.0,9 до гос,5.1,64.0
4,"0952294866, дочка Дарина 0678360161",ж,ІІ-ІІІ,2022-02-12,97.0,NaN,5.1,82.0
5,0959922983 сестра Софія,ж,ІІ-ІІІ,2021-12-15,88.0,NaN,OK,75.0
6,"0998073702, дружина Наталія 0998382226",ч,ІІ-ІІІ,2022-04-23,96.0,7 дні до госп,5.1,77.0
7,"0637258771, син Сергій 0968413301",ж,І,2022-05-18,96.0,9 днів до госп,5.1,47.0


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [60]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Телефон      120 non-null    object        
 1   Стать        121 non-null    object        
 2   ЛН           121 non-null    object        
 3   Поступлення  120 non-null    datetime64[ns]
 4   spO2 з дот   102 non-null    float64       
 5   Т вище38     77 non-null     object        
 6   БРпр2        99 non-null     object        
 7   С/Я2         104 non-null    float64       
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 7.7+ KB


In [61]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Стать
    непорожніх 121 з 121 | унікальних 4 | типи: str×121
    приклади: ["'ч'", "'ж'", "'чол'", "'жін'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Поступлення
    непорожніх 120 з 121 | унікальних 93 | типи: Timestamp×120
    приклади: ["Timestamp('2022-02-24 00:00:00')", "Timestamp('2022-02-22 00:00:00')", "Timestamp('2021-12-30 00:00:00')", "Timestamp('2021-12-24 00:00:00')"]

--- spO2 з дот
    непорожніх 102 з 121 | унікальних 13 | типи: float×102
    приклади: ['np.float64(98.0)', 'np.float64(95.0)', 'np.float64(90.0)', 'np.float64(96.0)']

--- Т вище38
    непорожніх 77 з 121 | унікальних 28 | типи: str×52, int×25
    приклади: ["'4 дні до госп

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [62]:
ПАСПОРТ = {
    'Телефон':     ('ідентифікатор',
                    'Це контактний номер родича пацієнта, а не виміряна величина'),
    'Стать':       ('номінальна',
                    'Значення «ч»/«ж» це лише назви категорій без природного порядку, тому допустиме тільки порівняння на рівність'),
    'ЛН':          ('порядкова',
                    'Ступінь легеневої недостатності I < I-II < II < II-III < III < III-IV впорядкований за тяжкістю, але відстань між сусідніми ступенями не визначена й не рівна'),
    'Поступлення': ('інтервальна',
                    'Дата госпіталізації має сенс як різниця (скільки днів між датами), але нуль календаря умовний, тому відношення двох дат безсенсове'),
    'spO2 з дот':  ('відношень',
                    'Сатурація на кисневій підтримці - відсоток насиченого киснем гемоглобіну з фізичним нулем, тому різниці й відношення мають сенс'),
    'Т вище38':    ('абсолютна',
                    'Це кількість днів із температурою понад 38 °C - лічба з природним нулем і природною одиницею (один день)'),
    'БРпр2':       ('відношень',
                    'Прямий білірубін у повторному аналізі вимірюється в мкмоль/л з природним нулем, відношення і різниця мають сенс'),
    'С/Я2':        ('відношень',
                    'Частка сегментоядерних нейтрофілів у повторному аналізі крові — відсоток від кількості лейкоцитів з природним нулем, тож відношення мають сенс.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"Це контактний номер родича пацієнта, а не виміряна величина"
1,Стать,номінальна,Значення «ч»/«ж» це лише назви категорій без природного ...
2,ЛН,порядкова,Ступінь легеневої недостатності I < I-II < II < II-III <...
3,Поступлення,інтервальна,Дата госпіталізації має сенс як різниця (скільки днів мі...
4,spO2 з дот,відношень,Сатурація на кисневій підтримці - відсоток насиченого ки...
5,Т вище38,абсолютна,Це кількість днів із температурою понад 38 °C - лічба з ...
6,БРпр2,відношень,Прямий білірубін у повторному аналізі вимірюється в мкмо...
7,С/Я2,відношень,Частка сегментоядерних нейтрофілів у повторному аналізі ...


In [63]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [64]:
import re

LN_ORDER = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']


def normalize(column):
    s = df[column]
    if column == 'Стать':
        return s.replace({'чол': 'ч', 'жін': 'ж'})
    if column == 'ЛН':
        return pd.Series(pd.Categorical(s, categories=LN_ORDER, ordered=True), name=s.name)
    if column == 'Т вище38':
        def days(v):
            m = re.match(r'\s*(\d+)', str(v))
            return float(m.group(1)) if pd.notna(v) and m else np.nan
        return s.map(days)
    if column == 'БРпр2':
        return pd.to_numeric(s, errors='coerce')
    return s


def round_value(v):
    if isinstance(v, (float, np.floating)):
        return round(float(v), 2)
    if isinstance(v, pd.Timestamp):
        return v.date()
    if isinstance(v, pd.Timedelta):
        return f'{v / pd.Timedelta(days=1):.1f} дн.'
    return v


def допустима_статистика(стовпець, шкала):
    з = normalize(стовпець).dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    р['мода'] = з.mode().iloc[0]
    if ш in ('номінальна', 'порядкова'):
        р['частоти'] = з.value_counts(sort=(ш == 'номінальна')).to_dict()
    else:
        р['частоти (топ-5)'] = з.value_counts().head(5).to_dict()

    if ш == 'порядкова':
        codes = pd.Series(з.cat.codes)
        for label, q in (('Q1', 0.25), ('медіана', 0.5), ('Q3', 0.75)):
            р[label] = LN_ORDER[int(codes.quantile(q, interpolation='lower'))]
    elif ш != 'номінальна':
        р['Q1'], р['медіана'], р['Q3'] = з.quantile([0.25, 0.5, 0.75])

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        р['середнє'] = з.mean()
        р['ст. відхилення'] = з.std()

    if ш in ('відношень', 'абсолютна'):
        р['коеф. варіації, %'] = з.std() / з.mean() * 100

    return {k: round_value(v) for k, v in р.items()}


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Стать  (номінальна)
    {'непорожніх': 121, 'унікальних': 2, 'мода': 'ж', 'частоти': {'ж': 75, 'ч': 46}}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': 'ІІ-ІІІ', 'частоти': {'І': 12, 'І-ІІ': 9, 'ІІ': 21, 'ІІ-ІІІ': 59, 'ІІІ': 17, 'ІІІ-IV': 3}, 'Q1': 'ІІ', 'медіана': 'ІІ-ІІІ', 'Q3': 'ІІ-ІІІ'}

--- Поступлення  (інтервальна)
    {'непорожніх': 120, 'унікальних': 93, 'мода': datetime.date(2022, 4, 2), 'частоти (топ-5)': {Timestamp('2022-04-02 00:00:00'): 4, Timestamp('2022-04-05 00:00:00'): 4, Timestamp('2021-09-01 00:00:00'): 3, Timestamp('2021-11-16 00:00:00'): 3, Timestamp('2021-11-17 00:00:00'): 3}, 'Q1': datetime.date(2021, 11, 5), 'медіана': datetime.date(2022, 2, 27), 'Q3': datetime.date(2022, 4, 7), 'середнє': datetime.date(2022, 1, 12), 'ст. відхилення': '113.0 дн.'}

--- spO2 з дот  (відношень)
    {'непорожніх': 102, 'унікальних': 13, 'мода': 96.0, 'частоти (топ-5)': {96.0: 18, 97

    {'непорожніх': 104, 'унікальних': 45, 'мода': 82.0, 'частоти (топ-5)': {82.0: 6, 61.0: 5, 60.0: 5, 77.0: 5, 70.0: 5}, 'Q1': 61.0, 'медіана': 69.5, 'Q3': 79.25, 'середнє': 68.13, 'ст. відхилення': 15.25, 'коеф. варіації, %': 22.38}



---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [65]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Стать,121,4,ж,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Поступлення,120,NaN,NaN,NaN,2022-01-12 07:24:00,2021-02-25 00:00:00,2021-11-05 06:00:00,2022-02-27 12:00:00,2022-04-07 00:00:00,2022-06-03 00:00:00,NaN
spO2 з дот,102.0,NaN,NaN,NaN,94.294118,80.0,93.0,95.0,96.0,98.0,3.11365
Т вище38,77.0,28.0,3.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
БРпр2,99.0,16.0,5.1,38.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
С/Я2,104.0,NaN,NaN,NaN,68.131731,16.0,61.0,69.5,79.25,91.0,15.245717


In [66]:
# Телефон: 23 номери Excel зберіг як int - можна знайти їх середнє
phone_numeric = pd.to_numeric(df['Телефон'], errors='coerce')
phone_mean = phone_numeric.mean()

# Поступлення: коефіцієнт варіації дати залежить від того, звідки рахувати дні
admission = df['Поступлення'].dropna()
def date_cv(origin):
    days = (admission - pd.Timestamp(origin)).dt.days
    return days.std() / days.mean() * 100
cv_unix, cv_excel, cv_2021 = date_cv('1970-01-01'), date_cv('1899-12-30'), date_cv('2021-01-01')

# describe() по сирих стовпцях
describe_all = df[МОЇ].describe(include='all').T
s_unique = describe_all.loc['Стать', 'unique']
fever_unique, fever_top, fever_freq = describe_all.loc['Т вище38', ['unique', 'top', 'freq']]

print(f'Середнє номерів телефону ({phone_numeric.count()} числових із {df["Телефон"].count()}): {phone_mean:,.0f}')
print(f'CV дати поступлення: від 1970 — {cv_unix:.2f} %, від 1900 — {cv_excel:.2f} %, від 2021 — {cv_2021:.2f} %')
print(f'describe(): Стать — unique {s_unique}; Т вище38 — unique {fever_unique}, top {fever_top!r}, freq {fever_freq}')
print()

БЕЗГЛУЗДІ = [
    ('Телефон', 'середнє', round(phone_mean),
     f'Телефон — ідентифікатор: цифри номера є кодом абонента, а не кількістю чогось, тому «середній номер» '
     f'не належить нікому; до того ж його пораховано лише з {phone_numeric.count()} рядків, які Excel випадково '
     f'зберіг як числа, а решта номерів записана текстом з іменами родичів.'),
    ('Поступлення', 'коефіцієнт варіації, %', round(cv_unix, 2),
     f'Дата — інтервальна шкала з умовним нулем, а CV ділить розкид на середнє, тобто на відстань до довільно '
     f'обраного початку відліку: ті самі дати дають {cv_unix:.2f} % від 1970 р., {cv_excel:.2f} % від 1900 р. '
     f'(Excel) і {cv_2021:.2f} % від 2021 р.'),
    ('Стать', 'describe: unique', s_unique,
     'Статей у даних дві, а 4 — артефакт запису: «ч»/«чол» і «ж»/«жін» — ті самі категорії; з тієї ж причини '
     'top=«ж» з freq=56 занижує справжню частоту жінок (75).'),
    ('Т вище38', 'describe: unique / top / freq', f'{fever_unique} / {fever_top} / {fever_freq}',
     'Стовпець — лічба днів із температурою, але pandas вважає «4 дні до госп», «4дні до госп» і число 4 '
     'різними категоріями: 28 «унікальних» значень і мода 3 (10 разів) хибні, насправді мода — 4 дні (23 рядки).'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

Середнє номерів телефону (23 числових із 120): 758,798,950
CV дати поступлення: від 1970 — 0.59 %, від 1900 — 0.25 %, від 2021 — 30.04 %
describe(): Стать — unique 4; Т вище38 — unique 28.0, top 3.0, freq 10.0



,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє,758798950,"Телефон — ідентифікатор: цифри номера є кодом абонента, ..."
1,Поступлення,"коефіцієнт варіації, %",0.59,"Дата — інтервальна шкала з умовним нулем, а CV ділить ро..."
2,Стать,describe: unique,4,"Статей у даних дві, а 4 — артефакт запису: «ч»/«чол» і «..."
3,Т вище38,describe: unique / top / freq,28.0 / 3.0 / 10.0,"Стовпець — лічба днів із температурою, але pandas вважає..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [67]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: spO2 без дот × помер/жив

spO2 без дот: шкала — не у вашому списку
помер/жив: шкала — не у вашому списку

   spO2 без дот  помер/жив
0          90.0          1
1          83.0          2
2          78.0          2
3          82.0          2
4          80.0          2
5          72.0          1


In [68]:
from scipy import stats

pair_scales = {A: 'відношень', B: 'номінальна (бінарна)'}
for col, scale in pair_scales.items():
    print(f'{col}: шкала — {scale}')
print()

ОБРАНА_МІРА = 'точковий бісеріальний коефіцієнт кореляції'
ЧОМУ_САМЕ_ВОНА = ('spO2 без дот — сатурація у відсотках, шкала відношень із природним нулем, а помер/жив — '
                  'бінарна номінальна змінна (1 — помер, 2 — жив); для пари відношень × бінарна номінальна '
                  'підходить точковий бісеріальний коефіцієнт, який по суті порівнює середню сатурацію двох груп.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

spo2 = df[A]
spo2_fixed = spo2.where(~(spo2 <= 1), spo2 * 100)

survived = (df[B] == 2).astype(int)        # 1 — вижив, 0 — помер
outcome = df[B].map({1: 'помер', 2: 'жив'})
mask = spo2_fixed.notna()

r_pb, p_pb = stats.pointbiserialr(survived[mask], spo2_fixed[mask])

print(f'r_pb = {r_pb:.3f}, n = {mask.sum()}')
print()
print(f'Висновок: зв’язок помірний і додатний — у пацієнтів, які вижили, сатурація без кисню при поступленні '
      f'в середньому вища; r_pb² = {r_pb**2:.2f}, тобто розподіл на групи пояснює близько {r_pb**2:.0%} '
      f'дисперсії spO2. Це зв’язок, а не причинність: низька сатурація — маркер тяжкості стану.')

spO2 без дот: шкала — відношень
помер/жив: шкала — номінальна (бінарна)

r_pb = 0.442, n = 120

Висновок: зв’язок помірний і додатний — у пацієнтів, які вижили, сатурація без кисню при поступленні в середньому вища; r_pb² = 0.19, тобто розподіл на групи пояснює близько 19% дисперсії spO2. Це зв’язок, а не причинність: низька сатурація — маркер тяжкості стану.


In [69]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.

r_12, _ = stats.pearsonr(df[B][mask], spo2_fixed[mask])
r_21, _ = stats.pearsonr(df[B][mask].map({1: 2, 2: 1}), spo2_fixed[mask])
print('Пірсон spO2 без дот × помер/жив')
print(f'   коди 1 = помер, 2 = жив:  r = {r_12:+.3f}')
print(f'   коди 1 = жив,   2 = помер: r = {r_21:+.3f}')
print('   Для двох груп число збігається з r_pb за модулем, але знак визначає довільний вибір кодів,')
print('   а «лінійне зростання помер/жив на одиницю» нічого не означає. Без знання кодування число не прочитати,')
print('   а для трьох і більше категорій (наприклад, Стать із 4 сирими мітками) воно втрачає зміст повністю.')
print()

Пірсон spO2 без дот × помер/жив
   коди 1 = помер, 2 = жив:  r = +0.442
   коди 1 = жив,   2 = помер: r = -0.442
   Для двох груп число збігається з r_pb за модулем, але знак визначає довільний вибір кодів,
   а «лінійне зростання помер/жив на одиницю» нічого не означає. Без знання кодування число не прочитати,
   а для трьох і більше категорій (наприклад, Стать із 4 сирими мітками) воно втрачає зміст повністю.



---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

**Чому всі числові стовпці стандартизуємо, а не нормалізуємо**

- Модель у лабораторній не задана. Стандартизація — типовий вибір для моделей, чутливих
  до масштабу (kNN, SVM, PCA, регресія з регуляризацією); деревам масштабування не потрібне.
- Один спосіб для всіх ознак: після min-max значення лежать у [0, 1], після z-оцінки —
  приблизно в [−3, 3]. Змішані, вони отримали б різну вагу у відстанях.
- Min-max чутлива до викидів: у `С/Я2` мінімум 16 при Q1 = 61, тож половина пацієнтів
  між Q1 і Q3 стиснулася б у смугу ≈ 0.60–0.84.
- Обидва перетворення лінійні й допустимі і для інтервальної шкали, і для шкали відношень:
  вибір між ними визначають модель і викиди, а не шкала.
- z-оцінка не змінює форму розподілу: правостороння асиметрія `Т вище38` (≈ 1.3) лишається.
- Середнє і σ рахуємо по всьому набору, бо поділу на train/test тут немає; у моделі їх
  обчислюють лише на тренувальній вибірці.

In [70]:
ПІДГОТОВКА = {
    'Телефон':     'не беремо в модель: ідентифікатор',
    'Стать':       'уніфікувати «чол»/«жін» → «ч»/«ж», далі one-hot, порядку не вводимо',
    'ЛН':          'впорядкована категорія І < І-ІІ < ІІ < ІІ-ІІІ < ІІІ < ІІІ-IV і цілі коди 0–5',
    'Поступлення': 'перейти від дати до тривалості — днів від першого поступлення в наборі — і стандартизувати',
    'spO2 з дот':  'стандартизація (z-оцінка): відсоток зі шкали відношень',
    'Т вище38':    'витягти з тексту число днів, далі стандартизація (z-оцінка), як і решта числових; порожні клітинки не відкидати, поки не з’ясовано, чи це 0 днів',
    'БРпр2':       'привести до числа («OK» — норма, а не категорія), далі стандартизація (z-оцінка)',
    'С/Я2':        'стандартизація (z-оцінка), не min-max: викид 16 при Q1 = 61 стиснув би решту значень',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,не беремо в модель: ідентифікатор
1,Стать,номінальна,"уніфікувати «чол»/«жін» → «ч»/«ж», далі one-hot, порядку..."
2,ЛН,порядкова,впорядкована категорія І < І-ІІ < ІІ < ІІ-ІІІ < ІІІ < ІІ...
3,Поступлення,інтервальна,перейти від дати до тривалості — днів від першого поступ...
4,spO2 з дот,відношень,стандартизація (z-оцінка): відсоток зі шкали відношень
5,Т вище38,абсолютна,"витягти з тексту число днів, далі стандартизація (z-оцін..."
6,БРпр2,відношень,"привести до числа («OK» — норма, а не категорія), далі с..."
7,С/Я2,відношень,"стандартизація (z-оцінка), не min-max: викид 16 при Q1 =..."


In [71]:
def zscore(s):
    assert s.std() > 0, f'{s.name}: сталий стовпець, стандартизація дасть ділення на нуль'
    return (s - s.mean()) / s.std()


prepared = pd.DataFrame(index=df.index)

# Порядкова: ЛН → цілі коди в заданому порядку LN_ORDER
prepared['ЛН_код'] = normalize('ЛН').cat.codes

# Інтервальна: дата → дні від першого поступлення → z-оцінка
admission_days = (df['Поступлення'] - df['Поступлення'].min()).dt.days
prepared['поступлення_дні_z'] = zscore(admission_days)

# Відношень і абсолютна: z-оцінка; Т вище38 і БРпр2 спершу приводимо до числа
for c in ('spO2 з дот', 'Т вище38', 'БРпр2', 'С/Я2'):
    prepared[f'{c}_z'] = zscore(normalize(c))

print('Підготовлено ознак:', prepared.shape[1], '| Телефон виключено')
print()
print(prepared.describe().T[['count', 'mean', 'std', 'min', 'max']].round(2).to_string())
prepared.head(8)

Підготовлено ознак: 6 | Телефон виключено

                   count  mean  std   min   max
ЛН_код             121.0  2.57  1.2  0.00  5.00
поступлення_дні_z  120.0 -0.00  1.0 -2.84  1.25
spO2 з дот_z       102.0 -0.00  1.0 -4.59  1.19
Т вище38_z          77.0 -0.00  1.0 -1.74  3.12
БРпр2_z             98.0 -0.00  1.0 -2.12  3.15
С/Я2_z             104.0 -0.00  1.0 -3.42  1.50


,ЛН_код,поступлення_дні_z,spO2 з дот_z,Т вище38_z,БРпр2_z,С/Я2_z
0,5,0.377642,1.190205,-0.119080,0.087834,0.384913
1,3,0.359950,0.226706,0.420283,-1.121513,0.712874
2,2,-0.117723,-1.379127,3.117099,1.297181,1.368795
3,5,-0.170797,0.547872,2.577736,0.087834,-0.271009
4,3,0.271492,0.869039,NaN,0.087834,0.909650
5,3,-0.250410,-2.021460,NaN,NaN,0.450505
6,3,0.890697,0.547872,1.499010,0.087834,0.581689
7,0,1.111842,0.547872,2.577736,0.087834,-1.386077


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [72]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [73]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [74]:
ВИПРАВЛЕННЯ = [
    (1,
     'помер/жив - бінарна номінальна змінна: 1 і 2 це назви груп, а середнє для номінальної шкали недопустиме',
     '1.50 - не стан посередині між смертю й одужанням, а артефакт кодування, проміжного результату лікування не існує',
     'Частоти й мода: померли 61 зі 121 (50.4 %), вижили 60 (49.6 %), тобто летальність у відділенні 50.4 %'),
    (2,
     'Стать - номінальна шкала, але мітки не уніфіковано: «ч»/«чол» і «ж»/«жін» - ті самі категорії, '
     'тому nunique рахує варіанти запису, а не статі',
     '«4 статі» і «найбільша група - ж, 56 осіб (46 %)» описують введення даних: насправді жінок 75 (62 %), '
     'чоловіків 46 (38 %), а окремих груп «жін» і «чол» не існує',
     'Спершу уніфікувати мітки (чол -> ч, жін -> ж), потім value_counts'),
    (3,
     'ступінь КТ — порядкова шкала: sort_index сортує за алфавітом, мітки з різним записом, а середнє кодів для порядкової шкали недопустиме',
     '«Найлегший — важкий, найважчий — середній» — це алфавітний порядок; коди 0–11 роздано за алфавітом разом '
     'із дублікатами, тому 5.9 → «легкий» не пов’язане з тяжкістю й змінюється від будь-якої опечатки',
     'Уніфікувати мітки, задати порядок легкий < середній < важкий < вкрай важкий < критичний і рахувати '
     'частоти, моду й медіану'),
    (4,
     'КТ — шкала відношень, але стовпець змішаний: 27 часток (0.3) і 17 текстових записів-діапазонів (70-80%); '
     'to_numeric(errors="coerce") викидає весь текст',
     '39.6 % пораховано лише з 27 із 44 заповнених клітинок, причому викинуто саме описові записи — вибірка '
     'зміщена, а кількість спостережень у звіті не вказано',
     'Частки перевести у відсотки'),
    (5,
     'spO2 без дот — відсотки на шкалі відношень, але одне значення 0.9 записане часткою; припущення, що всі '
     'значення в одних одиницях, хибне, а середнє нестійке до такого викиду',
     'Мінімум 0.9 % у живої людини неможливий — це 90 %, і помилка тягне середнє вниз; '
     'до того ж «у більшості пацієнтів» — твердження про медіану, а не про середнє',
     'Перевірити діапазон, виправити 0.9 → 90 і звітувати медіану з квартилями'),
    (6,
     'Дата поступлення — інтервальна шкала з умовним нулем, а коефіцієнт варіації підходить для шкали відношень ',
     '0.595 % — це розкид дат, поділений на відстань до 1970-01-01; від іншого початку відліку CV зовсім інший, '
     'тож про однорідність потоку число нічого не каже',
     'Описувати розкид у днях: поступлення охоплюють 463 дні (25.02.2021–03.06.2022); '
     'однорідність потоку оцінювати за кількістю госпіталізацій по місяцях'),
    (7,
     'Стать × помер/жив — дві номінальні змінні; кореляцію Пірсона обрано за зручністю, а «лінійний зв’язок» '
     'між категоріями змісту не має',
     'Для таблиці 2×2 |r| = 0.177 збігається з V Крамера, але знак залежить від кодування, а «близьке до нуля» — '
     'оцінка без перевірки: летальність жінок 57 %, чоловіків 39 %, χ² = 3.78, p = 0.052 — слабкий зв’язок '
     'на межі значущості, а не його відсутність',
     'Таблиця спряженості, χ²-тест і коефіцієнт Крамера'),
    (8,
     'Min-max застосовано до всіх числових стовпців без огляду на шкалу й варіативність: нормалізовано номінальний '
     'код помер/жив, а ФЕРИТ2 має лише одне значення (max = min)',
     '«Усі значення приведено до [0, 1]» хибно: для ФЕРИТ2 0/0 дало мовчазний NaN (count = 0), а 0/1 для '
     'помер/жив — просто перейменовані групи; частина стовпців майже порожня',
     'Нормалізувати лише змінні інтервальної шкали й шкали відношень, попередньо перевіривши nunique > 1 і частку '
     'пропусків'),
    (9,
     'dropna() по всіх 77 стовпцях: припущення, що пропуски поодинокі, хибне — у кожному рядку щось порожнє ',
     'Стало 0 рядків: «повністю заповнений набір» — порожня таблиця, на якій модель не побудувати; висновок '
     'звіту прямо суперечить виведеному числу',
     'Спершу відкинути стовпці з великою часткою пропусків (наприклад, > 30 %), потім рядки — лишається 58 '
     'стовпців і 52 рядки; або імпутувати; і завжди перевіряти розмір після очищення'),
    (10,
     'Дата розтину — подія після результату (є лише в померлих) це витік цілі, а не ознака',
     'r = 1.00 означає лише «розтин роблять померлим». Крім того, r = 0.62 для ІЛ6-2 пораховано лише на 6 пацієнтах',
     'Виключити Дата розтину та інші постфактум-ознаки, відкидати стовпці з малою '
     'кількістю спостережень'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,помер/жив - бінарна номінальна змінна: 1 і 2 це назви гр...,"1.50 - не стан посередині між смертю й одужанням, а арте...","Частоти й мода: померли 61 зі 121 (50.4 %), вижили 60 (4..."
1,2,"Стать - номінальна шкала, але мітки не уніфіковано: «ч»/...","«4 статі» і «найбільша група - ж, 56 осіб (46 %)» описую...","Спершу уніфікувати мітки (чол -> ч, жін -> ж), потім val..."
2,3,ступінь КТ — порядкова шкала: sort_index сортує за алфав...,"«Найлегший — важкий, найважчий — середній» — це алфавітн...","Уніфікувати мітки, задати порядок легкий < середній < ва..."
3,4,"КТ — шкала відношень, але стовпець змішаний: 27 часток (...","39.6 % пораховано лише з 27 із 44 заповнених клітинок, п...",Частки перевести у відсотки
4,5,"spO2 без дот — відсотки на шкалі відношень, але одне зна...","Мінімум 0.9 % у живої людини неможливий — це 90 %, і пом...","Перевірити діапазон, виправити 0.9 → 90 і звітувати меді..."
5,6,"Дата поступлення — інтервальна шкала з умовним нулем, а ...","0.595 % — це розкид дат, поділений на відстань до 1970-0...",Описувати розкид у днях: поступлення охоплюють 463 дні (...
6,7,Стать × помер/жив — дві номінальні змінні; кореляцію Пір...,"Для таблиці 2×2 |r| = 0.177 збігається з V Крамера, але ...","Таблиця спряженості, χ²-тест і коефіцієнт Крамера"
7,8,Min-max застосовано до всіх числових стовпців без огляду...,"«Усі значення приведено до [0, 1]» хибно: для ФЕРИТ2 0/0...",Нормалізувати лише змінні інтервальної шкали й шкали від...
8,9,"dropna() по всіх 77 стовпцях: припущення, що пропуски по...",Стало 0 рядків: «повністю заповнений набір» — порожня та...,Спершу відкинути стовпці з великою часткою пропусків (на...
9,10,Дата розтину — подія після результату (є лише в померлих...,r = 1.00 означає лише «розтин роблять померлим». Крім то...,"Виключити Дата розтину та інші постфактум-ознаки, відкид..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [75]:
# Розділ 3: ступінь КТ — порядкова шкала, порядок задаємо самі
CT_GRADE_ORDER = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
CT_GRADE_FIX = {'легка': 'легкий', 'сердній': 'середній', 'серед ступ': 'середній', 'серед ступеня': 'середній'}

wrong_grade_codes = {name: i for i, name in enumerate(df['ступінь КТ'].value_counts().sort_index().index)}
wrong_grade_mean = df['ступінь КТ'].map(wrong_grade_codes).mean()

ct_grade = df['ступінь КТ'].str.strip().replace(CT_GRADE_FIX)
ct_grade = pd.Series(pd.Categorical(ct_grade, categories=CT_GRADE_ORDER, ordered=True))
ct_grade_codes = pd.Series(ct_grade.cat.codes[ct_grade.notna()])
ct_grade_median = CT_GRADE_ORDER[int(ct_grade_codes.quantile(0.5, interpolation='lower'))]

print('РОЗДІЛ 3. Ступінь ураження за КТ')
print(f'   було:  середнє алфавітних кодів 0–11 = {wrong_grade_mean:.2f} → «легкий»')
print(f'   стало: частоти {ct_grade.value_counts(sort=False).to_dict()}')
print(f'          мода — {ct_grade.mode()[0]}, медіана — {ct_grade_median} (n = {ct_grade.count()})')
print()


# Розділ 4: КТ — частки й текстові діапазони переводимо у відсотки, нічого не викидаючи
def ct_percent(v):
    if isinstance(v, (int, float)):
        return v * 100                                
    bounds = [float(x) for x in re.findall(r'\d+', str(v))]
    return np.mean(bounds) if bounds else np.nan         


ct_wrong = pd.to_numeric(df['КТ'], errors='coerce')
ct_fixed = df['КТ'].map(ct_percent)

print('РОЗДІЛ 4. Обсяг ураження легень')
print(f'   було:  n = {ct_wrong.count()}, середнє {100 * ct_wrong.mean():.1f} %, медіана {100 * ct_wrong.median():.1f} %'
      f'  (текстові записи викинуто мовчки)')
print(f'   стало: n = {ct_fixed.count()}, середнє {ct_fixed.mean():.1f} %, медіана {ct_fixed.median():.1f} %, '
      f'ст. відхилення {ct_fixed.std():.1f} %')
print()


# Розділ 10: без витоку цілі й з кількістю спостережень поруч із r
died = (df['помер/жив'] == 1).astype(int)
autopsy = pd.to_datetime(df['Дата розтину'], errors='coerce').astype('int64')
print('РОЗДІЛ 10. Рейтинг ознак для прогнозу летальності')
print(f'   було:  r(Дата розтину) = {abs(autopsy.corr(died)):.3f}')

features = df.select_dtypes('number').drop(columns='помер/жив')
ranking = pd.DataFrame({'r': features.corrwith(died), 'n': features.notna().sum()}).dropna()
ranking = ranking[ranking['n'] >= 60]                    # ІЛ6-2 (n = 6) і подібні — поза рейтингом
ranking = ranking.loc[ranking['r'].abs().sort_values(ascending=False).index].head(8)
print('   стало: Дата розтину виключено (подія після результату), стовпці з n < 60 відкинуто')
print(ranking.round(3).to_string())
print('   Знак r важливий: Лім2 < 0 — у померлих лімфоцитів менше. Показники «2» виміряні пізніше,')
print('   тож для прогнозу при поступленні чесніше брати лише перші аналізи.')

РОЗДІЛ 3. Ступінь ураження за КТ
   було:  середнє алфавітних кодів 0–11 = 5.90 → «легкий»
   стало: частоти {'легкий': 7, 'середній': 18, 'важкий': 10, 'вкрай важкий': 4, 'критичний': 3}
          мода — середній, медіана — середній (n = 42)

РОЗДІЛ 4. Обсяг ураження легень
   було:  n = 27, середнє 39.6 %, медіана 30.0 %  (текстові записи викинуто мовчки)
   стало: n = 43, середнє 42.5 %, медіана 35.0 %, ст. відхилення 22.0 %

РОЗДІЛ 10. Рейтинг ознак для прогнозу летальності
   було:  r(Дата розтину) = 1.000
   стало: Дата розтину виключено (подія після результату), стовпці з n < 60 відкинуто
           r    n
Лім2  -0.645  104
Л2     0.480  104
Вік    0.449  121
П/Я2   0.395  104
АЛАТ1  0.392  115
АСАТ1  0.388  114
АСАТ2  0.387  104
ШОЕ2   0.369  104
   Знак r важливий: Лім2 < 0 — у померлих лімфоцитів менше. Показники «2» виміряні пізніше,
   тож для прогнозу при поступленні чесніше брати лише перші аналізи.


---
# Етап 7. Висновки та індивідуальне питання

In [76]:
sex_counts = normalize('Стать').value_counts()
ln_mode = normalize('ЛН').mode()[0]
ln_mode_n = (normalize('ЛН') == ln_mode).sum()

ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
    f'Тип даних у pandas не збігається зі шкалою: Телефон (ідентифікатор) у {phone_numeric.count()} рядках '
    f'збережено як int, і з них легко порахувати безглуздий «середній номер» {phone_mean:,.0f}; Т вище38 '
    f'(лічба днів) і БРпр2 (шкала відношень) прочитано як текст через записи «4 дні до госп» та «OK», '
    f'а Стать має 4 мітки замість двох категорій (насправді жінок {sex_counts["ж"]}, чоловіків {sex_counts["ч"]}).',

    f'Шкала обмежує допустимі статистики: для порядкової ЛН коректні лише частоти, мода й медіана — обидві '
    f'«{ln_mode}» ({ln_mode_n} зі {len(df)} пацієнтів), а для інтервальної дати поступлення коефіцієнт варіації '
    f'не має сенсу — ті самі дати дають {cv_unix:.2f} % від 1970 р. і {cv_2021:.2f} % від 2021 р., тому розкид '
    f'дат описуємо в днях (ст. відхилення {admission.std().days} дн.).',

    f'Сатурація без кисню при поступленні помірно пов’язана з результатом лікування: точковий бісеріальний '
    f'r_pb = {r_pb:.2f} (p = {p_pb:.1e}, n = {mask.sum()}) після виправлення запису 0.9 → 90 %; Пірсон на кодах '
    f'1/2 дає те саме число за модулем, але знак змінюється від перекодування, тож міру зв’язку обирає пара шкал, '
    f'а не зручність.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Тип даних у pandas не збігається зі шкалою: Телефон (ідентифікатор) у 23 рядках збережено як int, і з них легко порахувати безглуздий «середній номер» 758,798,950; Т вище38 (лічба днів) і БРпр2 (шкала відношень) прочитано як текст через записи «4 дні до госп» та «OK», а Стать має 4 мітки замість двох категорій (насправді жінок 75, чоловіків 46).
2. Шкала обмежує допустимі статистики: для порядкової ЛН коректні лише частоти, мода й медіана — обидві «ІІ-ІІІ» (59 зі 121 пацієнтів), а для інтервальної дати поступлення коефіцієнт варіації не має сенсу — ті самі дати дають 0.59 % від 1970 р. і 30.04 % від 2021 р., тому розкид дат описуємо в днях (ст. відхилення 113 дн.).
3. Сатурація без кисню при поступленні помірно пов’язана з результатом лікування: точковий бісеріальний r_pb = 0.44 (p = 4.5e-07, n = 120) після виправлення запису 0.9 → 90 %; Пірсон на кодах 1/2 дає те саме число за модулем, але знак змінюється від перекодування, тож міру зв’язку обирає пара шкал, а не зручність.


In [77]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

# Кандидат: «spO2 з дот» — сатурація на кисні. Порожньо там, де кисню не давали
no_oxygen = df['spO2 з дот'].isna()
groups = no_oxygen.map({True: 'порожня', False: 'заповнена'})
mild = df['Перебіг'].str.strip() == 'серед важ'
ln_codes = normalize('ЛН').cat.codes

evidence = pd.DataFrame({
    'пацієнтів':               groups.value_counts(),
    'порожня «Дотація»':       df['Дотація'].isna().groupby(groups).sum(),
    'померло':                 died.groupby(groups).sum(),
    'летальність, %':          (died.groupby(groups).mean() * 100).round(1),
    'spO2 без дот, медіана':   spo2_fixed.groupby(groups).median(),
    'spO2 без дот, мін':       spo2_fixed.groupby(groups).min(),
    'spO2 без дот, макс':      spo2_fixed.groupby(groups).max(),
    'перебіг «серед важ»':     mild.groupby(groups).sum(),
    'найтяжча ЛН':             ln_codes.groupby(groups).max().map(dict(enumerate(LN_ORDER))),
}).T[['порожня', 'заповнена']]
print(evidence.to_string())
print()

oxygen_agree = (no_oxygen == df['Дотація'].isna()).sum()
mortality_all = died.mean()
mortality_after_dropna = died[~no_oxygen].mean()
mild_lost = (mild & no_oxygen).sum() / mild.sum()
rows_after_full_dropna = len(df[МОЇ].dropna())

ВІДПОВІДЬ = (
    f'Такий стовпець — «spO2 з дот» (сатурація на кисневій підтримці) з мого варіанта. Порожня клітинка в ньому\n '
    f'означає не «не виміряли», а «кисень не знадобився»: пропуск збігається з порожньою «Дотація» (тривалість \n'
    f'кисневої підтримки) в {oxygen_agree} зі {len(df)} рядків. У {no_oxygen.sum()} пацієнтів із пропуском сатурація \n'
    f'на повітрі при поступленні {evidence.loc["spO2 без дот, мін", "порожня"]:.0f}–'
    f'{evidence.loc["spO2 без дот, макс", "порожня"]:.0f} % (медіана '
    f'{evidence.loc["spO2 без дот, медіана", "порожня"]:.0f} % — норма), а в пацієнтів на кисні медіана '
    f'{evidence.loc["spO2 без дот, медіана", "заповнена"]:.0f} % і максимум '
    f'{evidence.loc["spO2 без дот, макс", "заповнена"]:.0f} %. Усі вони мають перебіг «серед важ», ЛН не вище '
    f'{evidence.loc["найтяжча ЛН", "порожня"]}, і жоден не помер (летальність 0 % проти '
    f'{evidence.loc["летальність, %", "заповнена"]} % у групі на кисні). Отже, пропуск — це найлегші пацієнти, '
    f'а не випадкова втрата даних.\n\n'
    f'dropna() тут небезпечний, бо викидає не шум, а цілу клінічну групу: після dropna(subset=["spO2 з дот"]) '
    f'летальність у вибірці «зростає» з {mortality_all * 100:.1f} % до {mortality_after_dropna * 100:.1f} %, зникає '
    f'{mild_lost * 100:.0f} % пацієнтів середньої тяжкості, а модель втрачає найсильнішу ознаку виживання (0 смертей у групі). '
    f'dropna() по всіх восьми моїх стовпцях лишає лише {rows_after_full_dropna} рядків зі {len(df)}. Середнє '
    f'spO2 з дот {df["spO2 з дот"].mean():.1f} % з етапу 2 описує тільки пацієнтів на кисні, '
    f'тому заповнювати пропуски ним теж хибно. Правильно — додати бінарну ознаку «потрібен кисень» = notna(), '
    f'а сатурацію для пацієнтів без дотації брати зі «spO2 без дот», бо вони дихали повітрям.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Знайдіть у файлі стовпець, у якому порожня клітинка означає не «невідомо», а «норма». Доведіть це числами й поясніть, чому dropna() тут небезпечний.

spO2 з дот            порожня заповнена
пацієнтів                  19       102
порожня «Дотація»          18         1
померло                     0        61
летальність, %            0.0      59.8
spO2 без дот, медіана    95.0      83.0
spO2 без дот, мін        90.0      50.0
spO2 без дот, макс       98.0      91.0
перебіг «серед важ»        19        11
найтяжча ЛН                ІІ    ІІІ-IV

Такий стовпець — «spO2 з дот» (сатурація на кисневій підтримці) з мого варіанта. Порожня клітинка в ньому
 означає не «не виміряли», а «кисень не знадобився»: пропуск збігається з порожньою «Дотація» (тривалість 
кисневої підтримки) в 119 зі 121 рядків. У 19 пацієнтів із пропуском сатурація 
на повітрі при поступленні 90–98 % (медіана 95 % — норма), а в пацієнтів на кисні медіана 83 % і максимум 91 %. Усі вони мають перебіг «серед 

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [78]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'синтаксис',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'ні',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірено код обчислень і побудови графіків, не знайдено помилок, але видалено деякі зайві частини (наприклад частини коду які не створюють жодної різниці або не потрібні у завданні)'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [79]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 9 готова до захисту.
Кшик Олена, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.